In [2]:
# ============================================================
# CONDITIONAL IMAGE COLORIZATION USING USER INPUT + GUI
# ============================================================


import cv2
import numpy as np
from tkinter import *
from tkinter import filedialog, colorchooser
from PIL import Image, ImageTk



image_path = None
gray_image = None
colorized_image = None

sky_color = (255, 0, 0)     # Default Blue (BGR)
grass_color = (0, 255, 0)   # Default Green (BGR)

# ------------------------------------------------------------
# LOAD IMAGE
# ------------------------------------------------------------

def load_image():
    global image_path, gray_image

    image_path = filedialog.askopenfilename(
        filetypes=[("Image Files", "*.jpg *.png *.jpeg")]
    )

    if image_path:
        gray_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        display_image(gray_image, original_label)

# ------------------------------------------------------------
# DISPLAY IMAGE
# ------------------------------------------------------------

def display_image(img, label):

    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (300, 300))

    pil_img = Image.fromarray(img)
    tk_img = ImageTk.PhotoImage(pil_img)

    label.config(image=tk_img)
    label.image = tk_img

# ------------------------------------------------------------
# CHOOSE SKY COLOR
# ------------------------------------------------------------

def choose_sky_color():
    global sky_color

    color = colorchooser.askcolor()[0]

    if color:
        # RGB to BGR
        sky_color = (int(color[2]), int(color[1]), int(color[0]))

# ------------------------------------------------------------
# CHOOSE GRASS COLOR
# ------------------------------------------------------------

def choose_grass_color():
    global grass_color

    color = colorchooser.askcolor()[0]

    if color:
        grass_color = (int(color[2]), int(color[1]), int(color[0]))

# ------------------------------------------------------------
# CONDITIONAL COLORIZATION
# ------------------------------------------------------------

def colorize_image():

    global gray_image, colorized_image

    if gray_image is None:
        return

    # Convert grayscale to BGR
    colorized_image = cv2.cvtColor(gray_image, cv2.COLOR_GRAY2BGR)

    h, w = gray_image.shape

    
    sky_region = colorized_image[0:h//2, :]
    grass_region = colorized_image[h//2:h, :]

    
    sky_overlay = np.full_like(sky_region, sky_color)
    grass_overlay = np.full_like(grass_region, grass_color)

    sky_result = cv2.addWeighted(sky_region, 0.4, sky_overlay, 0.6, 0)
    grass_result = cv2.addWeighted(grass_region, 0.4, grass_overlay, 0.6, 0)

    colorized_image[0:h//2, :] = sky_result
    colorized_image[h//2:h, :] = grass_result

    display_image(colorized_image, result_label)

# ------------------------------------------------------------
# SAVE IMAGE
# ------------------------------------------------------------

def save_image():

    global colorized_image

    if colorized_image is None:
        return

    save_path = filedialog.asksaveasfilename(
        defaultextension=".jpg",
        filetypes=[("JPEG", "*.jpg"), ("PNG", "*.png")]
    )

    if save_path:
        cv2.imwrite(save_path, colorized_image)

# ==========================================================
# GUI WINDOW
# ==========================================================

root = Tk()
root.title("Conditional Image Colorization")
root.geometry("1000x700")
root.configure(bg="#1e1e2f")

# ==========================================================
# TITLE
# ==========================================================

title = Label(
    root,
    text="🎨 Conditional Image Colorization",
    font=("Segoe UI", 24, "bold"),
    fg="white",
    bg="#1e1e2f"
)
title.pack(pady=15)

subtitle = Label(
    root,
    text="Choose custom colors and colorize grayscale images",
    font=("Segoe UI", 11),
    fg="lightgray",
    bg="#1e1e2f"
)
subtitle.pack()

# ==========================================================
# BUTTON FRAME
# ==========================================================

button_frame = Frame(root, bg="#1e1e2f")
button_frame.pack(pady=20)

button_style = {
    "font": ("Segoe UI", 11, "bold"),
    "width": 18,
    "height": 2,
    "bd": 0,
    "cursor": "hand2"
}

load_btn = Button(
    button_frame,
    text="📂 Load Image",
    command=load_image,
    bg="#3498db",
    fg="white",
    **button_style
)
load_btn.grid(row=0, column=0, padx=10, pady=10)

sky_btn = Button(
    button_frame,
    text="☁ Choose Sky Color",
    command=choose_sky_color,
    bg="#5dade2",
    fg="white",
    **button_style
)
sky_btn.grid(row=0, column=1, padx=10)


grass_btn = Button(
    button_frame,
    text="🌿 Choose Grass Color",
    command=choose_grass_color,
    bg="#27ae60",
    fg="white",
    **button_style
)
grass_btn.grid(row=0, column=2, padx=10)


colorize_btn = Button(
    button_frame,
    text="🎨 Colorize",
    command=colorize_image,
    bg="#f39c12",
    fg="white",
    **button_style
)
colorize_btn.grid(row=1, column=0, pady=15)


save_btn = Button(
    button_frame,
    text="💾 Save Output",
    command=save_image,
    bg="#e74c3c",
    fg="white",
    **button_style
)
save_btn.grid(row=1, column=1)


exit_btn = Button(
    button_frame,
    text="❌ Exit",
    command=root.destroy,
    bg="#7f8c8d",
    fg="white",
    **button_style
)
exit_btn.grid(row=1, column=2)

# ==========================================================
# IMAGE FRAME
# ==========================================================

image_frame = Frame(root, bg="#1e1e2f")
image_frame.pack(pady=25)


left_frame = Frame(
    image_frame,
    bg="#2c2c3e",
    padx=15,
    pady=15
)
left_frame.grid(row=0, column=0, padx=25)

right_frame = Frame(
    image_frame,
    bg="#2c2c3e",
    padx=15,
    pady=15
)
right_frame.grid(row=0, column=1, padx=25)


Label(
    left_frame,
    text="Original Image",
    font=("Segoe UI", 14, "bold"),
    fg="white",
    bg="#2c2c3e"
).pack()

Label(
    right_frame,
    text="Colorized Output",
    font=("Segoe UI", 14, "bold"),
    fg="white",
    bg="#2c2c3e"
).pack()


original_label = Label(
    left_frame,
    bg="black",
    width=300,
    height=300,
    relief="ridge",
    bd=3
)
original_label.pack(pady=10)

result_label = Label(
    right_frame,
    bg="black",
    width=300,
    height=300,
    relief="ridge",
    bd=3
)
result_label.pack(pady=10)

# ==========================================================
# STATUS BAR
# ==========================================================

status_label = Label(
    root,
    text="Ready",
    font=("Segoe UI", 10),
    fg="white",
    bg="#34495e",
    anchor="w"
)

status_label.pack(
    side=BOTTOM,
    fill=X
)

# ==========================================================
# RUN
# ==========================================================

root.mainloop()